# Cardiovascular mortality outcome validation

Link the NHANES 1999–2004 cohort to the CDC 2019 public use linked mortality files. Check the existing five-year all-cause label, then derive cardiovascular and other-cause death within five years. CDC leading cause codes `001` (diseases of heart) and `005` (cerebrovascular diseases) define cardiovascular death here. The model target remains five-year all-cause mortality.

In [1]:
from pathlib import Path
import pandas as pd

root = Path.cwd().resolve()
if root.name == "notebooks":
    root = root.parent
assert (root / "data/processed/cohort_9957.csv").exists(), "Run from the project root or notebooks directory"

cycles = ["1999_2000", "2001_2002", "2003_2004"]
frames = []
for cycle in cycles:
    path = root / "data/raw/mortality" / f"NHANES_{cycle}_MORT_2019_PUBLIC.dat"
    frame = pd.read_fwf(
        path,
        colspecs=[(0, 6), (14, 15), (15, 16), (16, 19), (42, 45)],
        names=["SEQN", "ELIGSTAT", "MORTSTAT", "UCOD_LEADING", "PERMTH_INT"],
        dtype={"UCOD_LEADING": "string"},
        na_values=["."],
    )
    frame["cycle"] = cycle
    frames.append(frame)
mortality = pd.concat(frames, ignore_index=True)
assert mortality.SEQN.is_unique
print(f"Raw mortality records: {len(mortality):,}")
print(mortality.MORTSTAT.value_counts(dropna=False).to_string())

Raw mortality records: 31,126
MORTSTAT
NaN    14084
0.0    12323
1.0     4719


In [2]:
cohort = pd.read_csv(root / "data/processed/cohort_9957.csv", usecols=["SEQN", "SDDSRVYR", "death_5yr"])
assert cohort.SEQN.is_unique and len(cohort) == 9957
linked = cohort.merge(mortality, on="SEQN", how="left", validate="one_to_one", indicator=True)
assert linked._merge.eq("both").all(), "Cohort participant missing from mortality files"
assert linked.MORTSTAT.isin([0, 1]).all(), "Cohort participant is ineligible for linked mortality"
assert linked.PERMTH_INT.notna().all(), "Missing follow-up duration"
assert linked.loc[linked.MORTSTAT.eq(0), "PERMTH_INT"].ge(60).all(), "Survivor censored before five years"

linked["death_5yr_rebuilt"] = linked.MORTSTAT.eq(1) & linked.PERMTH_INT.le(60)
assert linked.death_5yr_rebuilt.astype(int).eq(linked.death_5yr).all(), "Stored outcome differs from CDC linkage"
linked["cvd_death_5yr"] = linked.death_5yr_rebuilt & linked.UCOD_LEADING.isin(["001", "005"])
linked["other_death_5yr"] = linked.death_5yr_rebuilt & ~linked.cvd_death_5yr
assert (linked.cvd_death_5yr.astype(int) + linked.other_death_5yr.astype(int)).eq(linked.death_5yr).all()

print(f"Matched cohort participants: {len(linked):,}")
print(f"Five-year all-cause deaths: {linked.death_5yr_rebuilt.sum():,}")
print(f"Five-year cardiovascular deaths: {linked.cvd_death_5yr.sum():,}")
print(f"Five-year other-cause deaths: {linked.other_death_5yr.sum():,}")

Matched cohort participants: 9,957
Five-year all-cause deaths: 1,293
Five-year cardiovascular deaths: 486
Five-year other-cause deaths: 807


In [3]:
summary = linked.groupby("cycle", sort=True).agg(
    participants=("SEQN", "size"),
    all_cause_deaths=("death_5yr_rebuilt", "sum"),
    cardiovascular_deaths=("cvd_death_5yr", "sum"),
    other_deaths=("other_death_5yr", "sum"),
    minimum_survivor_followup=("PERMTH_INT", lambda x: x[linked.loc[x.index, "MORTSTAT"].eq(0)].min()),
)
display(summary)
print("Five-year leading cause codes among decedents:")
display(linked.loc[linked.death_5yr_rebuilt, "UCOD_LEADING"].value_counts(dropna=False).rename_axis("code").to_frame("deaths"))

,participants,all_cause_deaths,cardiovascular_deaths,other_deaths,minimum_survivor_followup
cycle,,,,,
1999_2000,3182,397,165,232,228.0
2001_2002,3483,441,157,284,203.0
2003_2004,3292,455,164,291,179.0


Five-year leading cause codes among decedents:


,deaths
code,
001,398
002,307
010,273
005,88
003,65
008,39
007,33
009,33
006,30


The checks above confirm complete linkage, at least 60 months of follow-up for people labeled as survivors, and exact agreement between the stored five-year label and the CDC mortality files. Cardiovascular and other-cause counts are descriptive competing outcomes; the four prediction models in notebook 06 use all-cause death.